# Shipment analytics walkthrough
Synthetic portfolio dataset, snapshot 1 January 2026 UTC. Run the pipeline first. This notebook reads the generated exports and does not mutate the database.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
df = pd.read_csv(ROOT / 'data/processed/shipment_performance.csv', parse_dates=['dispatch_at','promised_delivery_at','actual_delivery_at'])
df.head()

## Data quality
Undelivered shipments legitimately have missing outcomes. Inspect rejection reasons separately.

In [ ]:
df.info()
print(df.isna().sum())
quality = json.loads((ROOT / 'reports/data_quality.json').read_text())
quality

## Calculate KPIs
The denominator is delivered shipments; unresolved shipments are not on-time.

In [ ]:
delivered = df[df.status.eq('Delivered')]
{'shipments': len(df), 'delivered': len(delivered), 'on_time_rate': 1-delivered.is_delayed.mean(), 'mean_delay_hours': delivered.delay_hours.mean()}

## Carrier comparisons
Show volumes and confidence intervals. Route and service mix can confound these unadjusted rates.

In [ ]:
carriers = pd.read_csv(ROOT / 'data/processed/carrier_performance.csv')
carriers.sort_values('delay_rate')

In [ ]:
carriers.sort_values('delay_rate').plot.barh(x='carrier_name', y='delay_rate', legend=False, title='Synthetic carrier delay rates')
plt.xlabel('Fraction of delivered shipments delayed')
plt.show()

## Seasonal patterns
Group by dispatch month. Recent cohorts are incomplete.

In [ ]:
monthly = delivered.assign(month=delivered.dispatch_at.dt.to_period('M')).groupby('month').is_delayed.mean()
(1-monthly).plot(marker='o', figsize=(12,4), title='On-time delivery rate by dispatch cohort')
plt.show()

## Route and service comparisons

In [ ]:
delivered.groupby(['route_id','service_level','carrier_name']).agg(delivered=('shipment_id','size'), delay_rate=('is_delayed','mean')).query('delivered >= 30').sort_values(['route_id','service_level','delay_rate']).head(20)

## Model evaluation
Read validation candidates separately from the one held-out test result. Synthetic results do not establish real-world accuracy.

In [ ]:
metrics = json.loads((ROOT / 'reports/model_metrics.json').read_text())
print('Selected model:', metrics['selected_model'])
print('Temporal split sizes:', metrics['split_rows'])
pd.DataFrame(metrics['validation_candidates']).T

In [ ]:
metrics['test_metrics']

## Next exercise
Explain which columns would leak future information at dispatch. Then inspect `src/model.py` and the temporal split tests. Assemble the native Power BI report using `powerbi/BUILD_GUIDE.md`.